# Performance: Indexes and Scans

In [1]:
# Run this cell to set up imports
import numpy as np
import pandas as pd

In [2]:
%reload_ext sql

## New IMDB Performance database

This is a variation of the IMDB database with keys defined. Note that this is a pretty big database! So if you run the below lines, please also remember to delete the `imdb_perf_lecture` afterwards to save space on your limited postgreSQL server.

In [3]:
!unzip -u data/imdb_perf_lecture.zip -d data/

Archive:  data/imdb_perf_lecture.zip
  inflating: data/imdb_perf_lecture.sql  


In [4]:
!psql -h localhost -c 'DROP DATABASE IF EXISTS imdb_perf_lecture'
!psql -h localhost -c 'CREATE DATABASE imdb_perf_lecture' 
!psql -h localhost -d imdb_perf_lecture -f data/imdb_perf_lecture.sql

NOTICE:  database "imdb_perf_lecture" does not exist, skipping
DROP DATABASE
CREATE DATABASE
SET
SET
SET
SET
SET
 set_config 
------------
 
(1 row)

SET
SET
SET
SET
SET
SET
CREATE TABLE
ALTER TABLE
CREATE TABLE
ALTER TABLE
CREATE TABLE
ALTER TABLE
COPY 845888
COPY 2211936
COPY 656453
ALTER TABLE
ALTER TABLE
ALTER TABLE
ALTER TABLE


In [5]:
%sql postgresql://127.0.0.1:5432/imdb_perf_lecture

Connecting to 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

## Display indexes

In [6]:
%sqlcmd tables

Name
movies
cast_info
actors


In [7]:
%sqlcmd columns -t actors

name,type,nullable,default,autoincrement,comment
id,INTEGER,False,None,False,None
name,TEXT,True,None,False,None


The meta-command `\d <relation>` shows indexes maintained with the `<relation>` table.

You can also look in the system view `pg_indexes` ([documentation 54.11](https://www.postgresql.org/docs/current/view-pg-indexes.html)):

In [9]:
%%sql
SELECT *
FROM pg_indexes
WHERE schemaname = 'public';

Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

2 rows affected.

schemaname,tablename,indexname,tablespace,indexdef
public,actors,actor_pkey,None,CREATE UNIQUE INDEX actor_pkey ON public.actors USING btree (id)
public,movies,movie_pkey,None,CREATE UNIQUE INDEX movie_pkey ON public.movies USING btree (id)


Read the `indexdef` as: the Actor relation has an index named `actor_pkey` which is created on the attribute `id`. In this case, the attribute `id` is also the **primary key** of the Actor relation, hence why it has an index. More on why primary keys automatically generate indexes in a bit.

# `EXPLAIN ANALYZE`

This query seems like it runs pretty quickly:

In [10]:
%%sql
SELECT * FROM Actors WHERE id = 23456;

Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

1 rows affected.

id,name
23456,Geraldo Alves


The PostgreSQL command `EXPLAIN ANALYZE` runs the **execution plan** of a statement and displays actual run time statistics. This is useful to understand what the query is actually doing. 

In [11]:
%%sql
EXPLAIN ANALYZE SELECT * FROM Actors WHERE id = 23456;

Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

4 rows affected.

QUERY PLAN
Index Scan using actor_pkey on actors (cost=0.42..8.44 rows=1 width=18) (actual time=0.017..0.018 rows=1 loops=1)
Index Cond: (id = 23456)
Planning Time: 0.064 ms
Execution Time: 0.032 ms


Try visualizing this on https://explain.dalibo.com/

<br/>

By contrast, the below query on `cast_info` runs quite slowly. Why?

In [12]:
%%sql
EXPLAIN ANALYZE SELECT * FROM Cast_info WHERE person_id = 23456;

Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

8 rows affected.

QUERY PLAN
Gather (cost=1000.00..22310.10 rows=16 width=8) (actual time=8.737..89.165 rows=3 loops=1)
Workers Planned: 2
Workers Launched: 2
-> Parallel Seq Scan on cast_info (cost=0.00..21308.50 rows=7 width=8) (actual time=32.609..84.860 rows=1 loops=3)
Filter: (person_id = 23456)
Rows Removed by Filter: 737311
Planning Time: 0.079 ms
Execution Time: 89.180 ms


<br/>

Explanation: `cast_info` does **not have an index** on `person_id`!


# Creating new Indexes

In the Actors table, `name` is not a primary key. What kind of scan do you think the following query will produce?

In [13]:
%sql EXPLAIN ANALYZE SELECT * FROM Actors WHERE name = 'Tom Hanks';

Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

8 rows affected.

QUERY PLAN
Gather (cost=1000.00..10631.77 rows=1 width=18) (actual time=0.313..24.006 rows=1 loops=1)
Workers Planned: 2
Workers Launched: 2
-> Parallel Seq Scan on actors (cost=0.00..9631.67 rows=1 width=18) (actual time=12.117..19.223 rows=0 loops=3)
Filter: (name = 'Tom Hanks'::text)
Rows Removed by Filter: 281962
Planning Time: 0.073 ms
Execution Time: 24.023 ms


We can manually create an index, even if it's not a primary key. Below, we create a multi-dimensional index just to show you the syntax:

In [14]:
%sql CREATE INDEX nameIdIndex ON actors(name,id);

Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

++
||
++
++

This makes our original query much faster:

In [15]:
%sql EXPLAIN ANALYZE SELECT * FROM actors WHERE name = 'Tom Hanks';

Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

5 rows affected.

QUERY PLAN
Index Only Scan using nameidindex on actors (cost=0.42..4.44 rows=1 width=18) (actual time=0.095..0.096 rows=1 loops=1)
Index Cond: (name = 'Tom Hanks'::text)
Heap Fetches: 0
Planning Time: 0.250 ms
Execution Time: 0.113 ms


Why "Index Only" Scan? Well, SQL correctly identified that there are only two attributes in the Actors table, and both are located in the index. So we just need to search the index; we don't need to additionally fetch any records.

# Exercise: Types of Scans

SQL automatically decides whether index scans are worth it. Sometimes, it decides to do a sequential scan instead, or even a bitmap heap scan.

<br/>

The below exact match lookup produces an Index Scan:

In [16]:
%sql EXPLAIN ANALYZE SELECT * FROM actors WHERE id = 23456;


Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

4 rows affected.

QUERY PLAN
Index Scan using actor_pkey on actors (cost=0.42..8.44 rows=1 width=18) (actual time=0.021..0.023 rows=1 loops=1)
Index Cond: (id = 23456)
Planning Time: 0.084 ms
Execution Time: 0.041 ms


This range lookup **also** produces an Index Scan:

In [17]:
%sql EXPLAIN ANALYZE SELECT * FROM actors WHERE 23456 <= id AND id < 23500;


Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

4 rows affected.

QUERY PLAN
Index Scan using actor_pkey on actors (cost=0.42..48.49 rows=14 width=18) (actual time=0.007..0.009 rows=11 loops=1)
Index Cond: ((id >= 23456) AND (id < 23500))
Planning Time: 0.127 ms
Execution Time: 0.020 ms


However, the below range lookup produces a **Sequential scan**!

In [18]:
%sql EXPLAIN ANALYZE SELECT * FROM actors WHERE id >= 23456;

Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

5 rows affected.

QUERY PLAN
Seq Scan on actors (cost=0.00..15799.60 rows=838222 width=18) (actual time=0.305..70.317 rows=838028 loops=1)
Filter: (id >= 23456)
Rows Removed by Filter: 7860
Planning Time: 0.087 ms
Execution Time: 98.128 ms


<br/>

And this other range lookup produces a **Bitmap Heap Scan**??

In [19]:
%sql EXPLAIN ANALYZE SELECT * FROM actors WHERE 5 <= id AND id < 23457;

Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

7 rows affected.

QUERY PLAN
Bitmap Heap Scan on actors (cost=166.99..5766.88 rows=7665 width=18) (actual time=0.598..1.590 rows=7857 loops=1)
Recheck Cond: ((5 <= id) AND (id < 23457))
Heap Blocks: exact=49
-> Bitmap Index Scan on actor_pkey (cost=0.00..165.08 rows=7665 width=0) (actual time=0.569..0.569 rows=7857 loops=1)
Index Cond: ((id >= 5) AND (id < 23457))
Planning Time: 0.106 ms
Execution Time: 1.857 ms



* Index scan:
    * For each index key match, there is a page fetch.
    * If multiple index key matches all correspond to a single page, that single page may get fetched multiple times.matches on our query.
* Sequential scan:
    * Once each page is loaded in, all records on that page are scanned in sequence.
* Bitmap heap scan:
    * Pre-scans the index to identify the unique pages to visit, then sequentially scans the subset of pages
    * More here: [stackoverflow](https://stackoverflow.com/questions/6592626/what-is-a-bitmap-heap-scan-in-a-query-plan)
    
<br/><br/><br/><br/>

Takeaway:
* There is no guarantee that records on disk are sorted in the same way as records in the index.
* Therefore index lookups are effectively random lookups! Many random lookups are typically more expensive than many sequential lookups!

<br/>
<br/>
Other range lookups for your practice:

In [25]:
%sql EXPLAIN ANALYZE SELECT * FROM actors WHERE id >= 23456 AND id < 23500;

Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

4 rows affected.

QUERY PLAN
Index Scan using actor_pkey on actors (cost=0.42..54.10 rows=16 width=18) (actual time=0.009..0.012 rows=11 loops=1)
Index Cond: ((id >= 23456) AND (id < 23500))
Planning Time: 0.132 ms
Execution Time: 0.025 ms


In [26]:
%sql EXPLAIN ANALYZE SELECT * FROM actors WHERE id >= 23456 AND id < 23457;

Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

4 rows affected.

QUERY PLAN
Index Scan using actor_pkey on actors (cost=0.42..8.45 rows=1 width=18) (actual time=0.007..0.009 rows=1 loops=1)
Index Cond: ((id >= 23456) AND (id < 23457))
Planning Time: 0.094 ms
Execution Time: 0.019 ms


In [27]:
%sql EXPLAIN ANALYZE SELECT * FROM actors WHERE id >= 23456 OR id < 23457;

Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

4 rows affected.

QUERY PLAN
Seq Scan on actors (cost=0.00..17921.32 rows=837665 width=18) (actual time=0.006..74.852 rows=845888 loops=1)
Filter: ((id >= 23456) OR (id < 23457))
Planning Time: 0.091 ms
Execution Time: 99.636 ms


# Cleanup

We drop the newly created index just to clean things up:

In [28]:
%sql DROP INDEX nameIdIndex;

Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

++
||
++
++

And we close the connection, then drop the database:

In [29]:
%sql --close postgresql://127.0.0.1:5432/imdb_perf_lecture

In [ ]:
!psql -h localhost -c 'DROP DATABASE IF EXISTS imdb_perf_lecture'